In [ ]:
import base64
import os
import time
from io import BytesIO
from pathlib import Path

import pypdfium2 as pdfium
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import (
    ChatCompletionContentPartParam,
    ChatCompletionUserMessageParam,
)

load_dotenv()

MODEL_NAME = "z-ai/glm-5.2"
URL = "https://openrouter.ai/api/v1"
API_KEY = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(base_url=URL, api_key=API_KEY)

In [2]:
def print_token_usage(usage, elapsed_time: float) -> None:
    if not usage:
        return

    prompt = usage.prompt_tokens
    completion = usage.completion_tokens
    total = usage.total_tokens
    speed = completion / elapsed_time if elapsed_time > 0 else 0.0

    completion_details = getattr(usage, "completion_tokens_details", None)
    reasoning = (
        (getattr(completion_details, "reasoning_tokens", 0) or 0)
        if completion_details
        else 0
    )
    output = completion - reasoning

    # Build key-value lines
    lines = [
        ("Prompt Tokens", f"{prompt:,}"),
        ("Completion Tokens", f"{completion:,}"),
    ]

    if reasoning > 0:
        pct_reasoning = (reasoning / completion) * 100
        pct_output = (output / completion) * 100
        lines.extend(
            [
                ("  ├─ Reasoning", f"{reasoning:,} ({pct_reasoning:.1f}%)"),
                ("  └─ Output", f"{output:,} ({pct_output:.1f}%)"),
            ]
        )

    lines.extend(
        [
            ("Total Tokens", f"{total:,}"),
            ("Speed", f"{speed:.2f} tok/sec"),
        ]
    )

    # Calculate exact inner width needed (minimum of 32 characters)
    inner_width = max(max(len(label) + len(val) for label, val in lines) + 4, 32)
    border_width = inner_width + 2

    # Draw the card
    print(f"\n┌{'─' * border_width}┐")
    print(f"│{'TOKEN USAGE'.center(border_width)}│")
    print(f"├{'─' * border_width}┤")

    for label, val in lines:
        # Separate the summary metrics (Total / Speed) with a line
        if label == "Total Tokens":
            print(f"├{'─' * border_width}┤")

        space_count = inner_width - len(label) - len(val)
        print(f"│ {label}{' ' * space_count}{val} │")

    print(f"└{'─' * border_width}┘")


def predict(
    prompt: str,
    file_path: str | Path | None = None,
    max_pages: int | None = None,
) -> tuple[str, str]:
    t0_total = time.perf_counter()
    content: list[ChatCompletionContentPartParam] = [{"type": "text", "text": prompt}]

    if file_path is not None:
        file_path = Path(file_path)
        suffix = file_path.suffix.lower()

        if suffix == ".pdf":
            print(f"[*] Rasterizing '{file_path}' with pypdfium2...")
            t0_pdf = time.perf_counter()

            pdf = pdfium.PdfDocument(str(file_path))
            page_count = len(pdf)
            page_limit = page_count if max_pages is None else min(max_pages, page_count)

            try:
                for page_index in range(page_limit):
                    page = pdf.get_page(page_index)
                    try:
                        image = page.render(scale=1).to_pil()
                    finally:
                        page.close()

                    buffered = BytesIO()
                    image.save(buffered, format="JPEG", quality=90)
                    img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

                    content.append(
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"},
                        }
                    )
            finally:
                pdf.close()

            t1_pdf = time.perf_counter()
            print(f"    -> PDF conversion took: {t1_pdf - t0_pdf:.2f} seconds.")
            print(f"    -> Prepared {page_limit} page image(s).")
        elif suffix in {".png", ".jpg", ".jpeg"}:
            print(f"[*] Loading image '{file_path}'...")
            t0_img = time.perf_counter()
            mime_type = "image/png" if suffix == ".png" else "image/jpeg"
            img_b64 = base64.b64encode(file_path.read_bytes()).decode("utf-8")
            content.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{mime_type};base64,{img_b64}"},
                }
            )
            t1_img = time.perf_counter()
            print(f"    -> Image load took: {t1_img - t0_img:.2f} seconds.")
        else:
            raise ValueError(
                f"Unsupported file type: '{suffix}'. Expected .pdf, .png, .jpg, or .jpeg."
            )

    print("[*] Sending request to LLM...")
    image_count = len(content) - 1
    if image_count > 0:
        print(f"    -> Found {image_count} image(s) to include in the request.")
    t0_llm = time.perf_counter()
    user_message: ChatCompletionUserMessageParam = {
        "role": "user",
        "content": content,
    }
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[user_message],
        extra_body={"reasoning": {"enabled": True}},
    )
    t1_llm = time.perf_counter()
    print(f"    -> inference took: {t1_llm - t0_llm:.2f} seconds.")
    print(f"    -> Total latency: {t1_llm - t0_total:.2f} seconds.")

    print_token_usage(response.usage, t1_llm - t0_llm)

    message = response.choices[0].message
    reasoning_content = getattr(message, "reasoning", None) or ""
    content_str = message.content or ""

    return reasoning_content, content_str

## First Test


In [4]:
prompt = """
I want to wash my car. The car wash is 50 meters away. Should I walk or drive?
"""

reasoning, response = predict(prompt=prompt)

[*] Sending request to LLM...
    -> inference took: 7.63 seconds.
    -> Total latency: 7.63 seconds.

┌──────────────────────────────────┐
│           TOKEN USAGE            │
├──────────────────────────────────┤
│ Prompt Tokens                 35 │
│ Completion Tokens            256 │
│   ├─ Reasoning       185 (72.3%) │
│   └─ Output           71 (27.7%) │
├──────────────────────────────────┤
│ Total Tokens                 291 │
│ Speed              33.57 tok/sec │
└──────────────────────────────────┘


In [5]:
print(reasoning)

1.  **Analyze the Input:**
    *   Goal: Wash the car.
    *   Distance to car wash: 50 meters.
    *   Question: Should I walk or drive?

2.  **Identify the Catch/Logical Fallacy:**
    *   If the user walks to the car wash, the car stays at home. Therefore, the car cannot be washed.
    *   If the user wants to wash the car, the car *must* go to the car wash.

3.  **Formulate the Response:**
    *   Point out the logical necessity of bringing the car.
    *   Humorous but clear tone.
    *   Direct answer: Drive (or push/tow the car).

4.  **Drafting the Response:**
    *   "You should definitely drive. If you walk to the car wash, you'll be there, but your car will still be at home!"


In [6]:
print(response)

You should definitely drive! 

If you walk to the car wash, you will be 50 meters away from your house, but your car will still be sitting dirty in the driveway. Unless you're planning to bring a bucket and sponge back to the car, you need to take the car with you!


## Vacuum Fall


In [7]:
prompt = """
I have 4 objects: A, B, C, and D. A is heavier than B. C is lighter than D. 
B is heavier than C. If I drop A and D at the exact same moment from a 
100-meter tower inside a giant vacuum chamber, which one hits the ground first? 
Explain your reasoning step-by-step.
"""
reasoning, response = predict(prompt=prompt)

[*] Sending request to LLM...
    -> inference took: 16.42 seconds.
    -> Total latency: 16.42 seconds.

┌──────────────────────────────────┐
│           TOKEN USAGE            │
├──────────────────────────────────┤
│ Prompt Tokens                 86 │
│ Completion Tokens            986 │
│   ├─ Reasoning       641 (65.0%) │
│   └─ Output          345 (35.0%) │
├──────────────────────────────────┤
│ Total Tokens               1,072 │
│ Speed              60.06 tok/sec │
└──────────────────────────────────┘


In [8]:
print(reasoning)

1.  **Analyze the Request:**
    *   Four objects: A, B, C, D.
    *   Conditions:
        *   A > B (weight/mass)
        *   C < D (weight/mass)
        *   B > C (weight/mass)
    *   Scenario: Drop A and D simultaneously from a 100-meter tower *inside a giant vacuum chamber*.
    *   Question: Which one hits the ground first?
    *   Constraint: Explain reasoning step-by-step.

2.  **Analyze the Scenario Physics:**
    *   The objects are dropped in a *vacuum chamber* (meaning no air resistance).
    *   According to Galileo's principle of equivalence and Newton's laws of motion, in a vacuum, all objects fall at the same rate of acceleration due to gravity ($g \approx 9.8 \text{ m/s}^2$), regardless of their mass or weight.
    *   Therefore, A and D will hit the ground at the exact same time.

3.  **Address the given weight conditions:**
    *   A > B
    *   C < D
    *   B > C
    *   These conditions establish a relative weight ordering: A > B > C and D > C. But the exact relat

In [9]:
print(response)

Here is the step-by-step reasoning:

**Step 1: Analyze the weight clues**
Based on the information provided, we can determine the relative weights of some of the objects: 
* A is heavier than B (A > B)
* D is heavier than C (D > C)
* B is heavier than C (B > C)
From this, we know A and D are both heavier than C. However, we don't actually know the exact weight relationship between A and D. As we will see in the next steps, this doesn't matter.

**Step 2: Identify the environment**
The prompt specifies that the objects are dropped "inside a giant vacuum chamber." A vacuum chamber is an environment completely devoid of air. This means there is zero air resistance acting on the objects as they fall.

**Step 3: Apply the laws of physics**
When objects fall in a vacuum (where air resistance is eliminated), the only force acting upon them is gravity. According to the laws of physics—specifically Galileo's principle of falling bodies and Newton's laws of motion—the acceleration of a falling o

## SVG


In [3]:
prompt = """
Create SVG with a penguin, at the powerlifting rack and being ready to deadlift. Animate it. HTML only.
"""
reasoning, response = predict(prompt=prompt)

[*] Sending request to LLM...
    -> inference took: 74.93 seconds.
    -> Total latency: 74.93 seconds.

┌──────────────────────────────────┐
│           TOKEN USAGE            │
├──────────────────────────────────┤
│ Prompt Tokens                 39 │
│ Completion Tokens          6,266 │
│   ├─ Reasoning        177 (2.8%) │
│   └─ Output        6,089 (97.2%) │
├──────────────────────────────────┤
│ Total Tokens               6,305 │
│ Speed              83.62 tok/sec │
└──────────────────────────────────┘


In [4]:
print(reasoning)

The user wants a penguin at a powerlifting rack, ready to deadlift, animated, HTML only. Let me build this with SVG inside HTML. The penguin should be positioned at a barbell on the floor in front of a power rack, in a deadlift setup stance (hips back, chest up, gripping the bar). I'll animate breathing, eye blinks, maybe bar wobble, chalk dust, etc.

Let me design:
- Power rack: two upright posts with safety bars, maybe a pull-up bar at top
- Loaded barbell on floor with plates (45lb style)
- Penguin: black body, white belly, orange beak/feet, in deadlift stance bent over gripping bar
- Animations: breathing motion, slight bar shake (effort), eye blink, sweat drop, chalk particles

Let me write the code.


In [ ]:
print(response)

Here's an animated scene of a determined little penguin psyching up for a heavy deadlift inside a power rack — chalk dust drifting, sweat dripping, bar trembling under tension.

```html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Penguin Powerlifter</title>
<style>
  html,body{margin:0;height:100%;background:#07060a;display:flex;justify-content:center;align-items:center;font-family:-apple-system,Segoe UI,sans-serif}
  .stage{width:min(95vw,920px);aspect-ratio:4/3;position:relative;border-radius:18px;overflow:hidden;box-shadow:0 30px 80px rgba(0,0,0,.6),0 0 0 1px rgba(255,255,255,.04)}
  svg{width:100%;height:100%;display:block;background:#0a0810}
  .caption{position:absolute;left:18px;bottom:14px;color:#ffb98a;font-size:12px;letter-spacing:.18em;text-transform:uppercase;opacity:.65;font-weight:600}
</style>
</head>
<body>
<div class="stage">
<svg viewBox="0 0 800 600" xmlns="http://www.w3.org/2000/svg" aria-label="A penguin ready to deadlift inside a power rac

## Web Design


In [8]:
prompt = """
Create a simple CV/resume website for ML/AI engineer. Use matrix-like vibe with minimalistic and techy-like vibe. 
Use simple and flat design for the whole app. Create a single HTML/JS/CSS file that would represent the whole website.
"""
reasoning, response = predict(prompt=prompt)

[*] Sending request to LLM...
    -> inference took: 176.56 seconds.
    -> Total latency: 176.56 seconds.

┌──────────────────────────────────┐
│           TOKEN USAGE            │
├──────────────────────────────────┤
│ Prompt Tokens                 66 │
│ Completion Tokens          8,829 │
│   ├─ Reasoning        170 (1.9%) │
│   └─ Output        8,659 (98.1%) │
├──────────────────────────────────┤
│ Total Tokens               8,895 │
│ Speed              50.01 tok/sec │
└──────────────────────────────────┘


In [9]:
print(reasoning)

The user wants a matrix-like, minimalistic, techy CV/resume website for an ML/AI engineer. Single HTML file. Let me design this with the matrix rain effect, monospace fonts, flat design, green-on-black aesthetic but make it readable and professional.

Key sections:
- Hero with name, title, matrix rain background
- About
- Skills (ML/AI specific)
- Experience
- Projects
- Education
- Contact

Let me build this with a terminal/matrix vibe. Flat design, minimal animations. I'll use monospace font like JetBrains Mono or Share Tech Mono.

Let me make it nice with:
- Matrix rain canvas in background (subtle)
- Terminal-style typing for hero
- Section navigation
- Flat cards with borders
- Green accent color on black background

Let me write it.


In [10]:
print(response)

```html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>kai.sato // ML Engineer</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@300;400;500;700&family=Space+Grotesk:wght@300;400;500;700&display=swap" rel="stylesheet">
<style>
  :root{
    --bg:#0a0e0a;
    --bg-2:#0d130d;
    --fg:#c8d6c8;
    --fg-dim:#6b7d6b;
    --accent:#4ade80;
    --accent-2:#22c55e;
    --warn:#facc15;
    --line:rgba(74,222,128,0.18);
    --line-soft:rgba(200,214,200,0.08);
  }
  *{box-sizing:border-box;margin:0;padding:0}
  html,body{background:var(--bg);color:var(--fg);font-family:'JetBrains Mono',monospace;font-size:14px;line-height:1.6;overflow-x:hidden}
  body{position:relative;min-height:100vh}

  /* matrix canvas */
  #matrix{position:fixed;inset:0;z

## Bug Hunting


In [15]:
prompt = """
Is this code correct? 
If not explain the issues and provide the fully refactored, production-ready version of the code that resolves the issue(s):

<code>
import asyncio

balance = 100
lock = asyncio.Lock()

async def deposit(amount):
    global balance
    await asyncio.sleep(0.1) # Simulate network/DB latency
    async with lock:
        temp = balance
        await asyncio.sleep(0.1)
        balance = temp + amount

async def withdraw_with_bonus(amount):
    global balance
    async with lock:
        if balance >= amount:
            balance -= amount
            # Apply a quick loyalty bonus deposit
            await deposit(10) 

async def main():
    print(f"Initial balance: {balance}")
    print("Attempting withdrawal with bonus...")
    try:
        await asyncio.wait_for(withdraw_with_bonus(50), timeout=2.0)
        print(f"Success! New balance: {balance}")
    except asyncio.TimeoutError:
        print("\n[ERROR] Execution timed out! A deadlock occurred.")

if __name__ == "__main__":
    asyncio.run(main())
</code>
"""
reasoning, response = predict(prompt=prompt)

[*] Sending request to LLM...
    -> inference took: 28.99 seconds.
    -> Total latency: 28.99 seconds.

┌──────────────────────────────────┐
│           TOKEN USAGE            │
├──────────────────────────────────┤
│ Prompt Tokens                246 │
│ Completion Tokens          1,928 │
│   ├─ Reasoning     1,256 (65.1%) │
│   └─ Output          672 (34.9%) │
├──────────────────────────────────┤
│ Total Tokens               2,174 │
│ Speed              66.50 tok/sec │
└──────────────────────────────────┘


In [16]:
print(reasoning)

The user is asking if the provided Python `asyncio` code is correct.
If not, I need to explain the issues and provide a refactored, production-ready version.

Looking at the code:
```python
import asyncio

balance = 100
lock = asyncio.Lock()

async def deposit(amount):
    global balance
    await asyncio.sleep(0.1) # Simulate network/DB latency
    async with lock:
        temp = balance
        await asyncio.sleep(0.1)
        balance = temp + amount

async def withdraw_with_bonus(amount):
    global balance
    async with lock:
        if balance >= amount:
            balance -= amount
            # Apply a quick loyalty bonus deposit
            await deposit(10) 

async def main():
    print(f"Initial balance: {balance}")
    print("Attempting withdrawal with bonus...")
    try:
        await asyncio.wait_for(withdraw_with_bonus(50), timeout=2.0)
        print(f"Success! New balance: {balance}")
    except asyncio.TimeoutError:
        print("
[ERROR] Execution timed out! A deadl

In [17]:
print(response)

No, the code is not correct. It contains a critical concurrency issue and a syntax error.

### Issues Explained

1. **Deadlock**: `asyncio.Lock` is **not reentrant**. In your code, `withdraw_with_bonus` acquires the `lock` and then calls `deposit`. The `deposit` function attempts to acquire the same `lock` a second time. Because the lock is already held by `withdraw_with_bonus`, `deposit` will wait indefinitely for the lock to be released. This creates a deadlock, which is why the execution times out.
2. **Syntax Error**: The string literal inside the `except` block spans multiple lines without using triple quotes. This will raise a `SyntaxError` when you try to run the code.
3. **Global State**: Using `global` variables for state management is generally discouraged in production code as it makes the code harder to test, scale, and maintain.

### Refactored, Production-Ready Code

To fix these issues, we can encapsulate the balance and the lock inside a class. We can then create an "un

## N-Body Gravity Simulation


In [18]:
prompt = """
Create a fully interactive N-Body gravity simulation in a single HTML file using vanilla JavaScript and CSS.
- Render it on a full-screen HTML5 canvas with a dark theme.
- Particles should attract each other using Newton's law of universal gravitation.
- Left-click and drag to launch new particles (drag distance controls velocity vector, click duration/hold controls mass and size).
- If two particles collide, they should merge, combining their mass and conserving momentum.
- Add subtle trail effects to the particles so viewers can see their orbits.
- Keep the code clean, modular, and modern. Output only the single HTML file containing CSS and JS.
"""
reasoning, response = predict(prompt=prompt)

[*] Sending request to LLM...
    -> inference took: 119.48 seconds.
    -> Total latency: 119.48 seconds.

┌──────────────────────────────────┐
│           TOKEN USAGE            │
├──────────────────────────────────┤
│ Prompt Tokens                144 │
│ Completion Tokens          8,151 │
│   ├─ Reasoning        511 (6.3%) │
│   └─ Output        7,640 (93.7%) │
├──────────────────────────────────┤
│ Total Tokens               8,295 │
│ Speed              68.22 tok/sec │
└──────────────────────────────────┘


In [19]:
print(reasoning)

The user wants a full N-body gravity simulation. Let me design this carefully.

Key features:
- Full-screen canvas, dark theme
- Newton's gravity between particles
- Click+drag to launch: drag distance = velocity, hold duration = mass/size
- Collision merges particles (mass + momentum conservation)
- Trail effects
- Clean modular code

Let me design the visual: deep space dark background with subtle nebula. Particles glow with color based on mass (small = blue-white, large = orange/red like stars). Trails fade behind.

Physics:
- F = G*m1*m2/r²
- Use softening to avoid singularities
- Sub-stepping for stability
- Spatial: simple O(n²) is fine for a few hundred particles

Let me think about the launch mechanic:
- mousedown starts timer & position
- mousemove tracks current
- mouseup launches particle with velocity proportional to drag vector, mass proportional to hold duration
- Show preview line + growing circle while dragging

Visual style: 
- Background: dark navy/black with subtle r

In [20]:
print(response)

Looking at this challenge, I want to build something that feels less like a physics demo and more like a cosmic playground — glowing stars with real gravitational choreography, soft fading trails, and a launch mechanic that's tactile and predictable.

Here's my approach:
- **Physics**: Velocity-Verlet-ish integration with sub-stepping and softening to keep orbits stable
- **Visuals**: Mass-mapped stellar colors (cool blue dwarfs → warm red giants), radial-gradient glows, fading polyline trails, static background starfield + nebula
- **Launch UX**: Live preview showing direction arrow and growing radius while you hold, so you can aim and "charge" mass before release
- **Collisions**: Inelastic merge conserving momentum, with a brief flash effect

```html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>N-Body Gravity — Cosmic Sandbox</title>
<style>
  :root {
    --bg-0: #05060d;
    --bg-1: 

## Game Generation


In [21]:
prompt = """
Generate ultra realistic water in three.js with a boat on it and occasional waves hitting it, real physics. 
I want to be able to move the boat and have enemies trying to kill my boat. 
I should be able to move the boat with keyboard and aim with the mouse. The result should be a single HTML file.
"""
reasoning, response = predict(prompt=prompt)

[*] Sending request to LLM...
    -> inference took: 264.02 seconds.
    -> Total latency: 264.02 seconds.

┌──────────────────────────────────┐
│           TOKEN USAGE            │
├──────────────────────────────────┤
│ Prompt Tokens                 79 │
│ Completion Tokens         11,702 │
│   ├─ Reasoning     1,992 (17.0%) │
│   └─ Output        9,710 (83.0%) │
├──────────────────────────────────┤
│ Total Tokens              11,781 │
│ Speed              44.32 tok/sec │
└──────────────────────────────────┘


In [22]:
print(reasoning)

The user wants an ultra-realistic water simulation in three.js with a boat, physics, keyboard movement, mouse aiming, and enemies. Single HTML file.

Let me plan this carefully:

1. **Water**: Use Three.js Water from examples (jsm/objects/Water.js) - it's reflective, realistic
2. **Boat**: A simple boat mesh, can be made from geometry. Add buoyancy physics - sample wave height at boat position
3. **Waves**: Use the Water shader with normals map, plus sine waves for physics
4. **Enemies**: Other boats/sharks that chase player. Player can shoot with mouse aim
5. **Controls**: WASD movement, mouse to aim, click to shoot
6. **Physics**: Buoyancy - sample water height, apply forces

Let me code this:

- Sky with sun
- Water with normal map waves
- Boat with buoyancy (4 corner sampling for tilt)
- Cannons that shoot cannonballs
- Enemy boats that chase and shoot
- Health bar
- HUD

For water height sampling, I'll use a Gerstner wave function that matches the visual water.

Let me write a com

In [23]:
print(response)

```html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Tidebreaker — Naval Combat</title>
<link href="https://fonts.googleapis.com/css2?family=Cinzel:wght@500;700;900&family=Rajdhani:wght@400;600;700&display=swap" rel="stylesheet">
<style>
  :root {
    --gold: #f4c95d;
    --crimson: #d94b4b;
    --deep: #0a1622;
    --foam: #e9f4ff;
  }
  * { box-sizing: border-box; }
  html, body { margin: 0; padding: 0; overflow: hidden; background: #000; height: 100%; }
  body { font-family: 'Rajdhani', sans-serif; color: var(--foam); cursor: crosshair; }
  canvas { display: block; }

  #hud {
    position: fixed; inset: 0; pointer-events: none; z-index: 10;
    padding: 22px 30px; display: flex; flex-direction: column; justify-content: space-between;
  }
  .top-bar, .bot-bar { display: flex; justify-content: space-between; align-items: flex-start; }
  .panel {
    background: linear-gradient(135deg, rgba(10,22,34,.78), rgba(20,40,60,.55));
    border: 1px solid rgba(244,201